In [ ]:
import os
import pandas as pd

folder_path = './dictionary_files/'
filepaths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.csv')]

df = pd.concat([pd.read_csv(filepath) for filepath in filepaths], ignore_index=True)
usage_df = df[['word', 'wordcount', 'bookcount']]
meaning_df = df.drop(columns=['word', 'wordcount', 'bookcount', 'error'])

meaning_df = meaning_df[~meaning_df['word.1'].isnull()].drop_duplicates(subset='word.1', keep='first')
# meaning_df = meaning_df[meaning_df['phonetics'] != '[]']

usage_df = usage_df.drop_duplicates(subset='word', keep='first')
final_df = meaning_df.merge(usage_df, left_on='word.1', right_on='word', how='left').drop(['word.1'], axis=1)
final_df = final_df[['word', 'wordcount', 'bookcount', 'meanings', 'phonetics', 'audio', 'sourceUrls']]

final_df.to_csv('lexodle_list.csv', index=False)

,word.1,phonetics,meanings,sourceUrls,audio
7,aback,"[{'text': '/əˈbæk/', 'audio': ''}, {'text': '/...","[{'partOfSpeech': 'adverb', 'definitions': [{'...",['https://en.wiktionary.org/wiki/aback'],./wordle/dictionary_audio/aback.mp3
9,abaft,"[{'text': '/əˈbæft/', 'audio': ''}, {'text': '...","[{'partOfSpeech': 'adverb', 'definitions': [{'...",['https://en.wiktionary.org/wiki/abaft'],./wordle/dictionary_audio/abaft.mp3
11,abate,"[{'text': '/əˈbeɪt/', 'audio': ''}, {'text': '...","[{'partOfSpeech': 'noun', 'definitions': [{'de...",['https://en.wiktionary.org/wiki/abate'],NaN
13,abbey,"[{'text': '/ˈæb.i/', 'audio': 'https://api.dic...","[{'partOfSpeech': 'noun', 'definitions': [{'de...",['https://en.wiktionary.org/wiki/abbey'],./wordle/dictionary_audio/abbey.mp3
15,abbot,"[{'text': '/ˈæb.ət/', 'audio': ''}, {'text': '...","[{'partOfSpeech': 'noun', 'definitions': [{'de...",['https://en.wiktionary.org/wiki/abbot'],./wordle/dictionary_audio/abbot.mp3
...,...,...,...,...,...
69230,shims,[],"[{'partOfSpeech': 'noun', 'definitions': [{'de...","['https://en.wiktionary.org/wiki/shim', 'https...",NaN
69231,shine,"[{'text': '', 'audio': 'https://api.dictionary...","[{'partOfSpeech': 'noun', 'definitions': [{'de...",['https://en.wiktionary.org/wiki/shine'],./wordle/dictionary_audio/shine.mp3
69234,shins,[],"[{'partOfSpeech': 'noun', 'definitions': [{'de...","['https://en.wiktionary.org/wiki/shin', 'https...",NaN
69235,shiny,"[{'text': '/ˈʃaɪni/', 'audio': 'https://api.di...","[{'partOfSpeech': 'noun', 'definitions': [{'de...",['https://en.wiktionary.org/wiki/shiny'],./wordle/dictionary_audio/shiny.mp3


In [3]:
import pandas as pd

def filter_words_by_absent_letters(filepath, absent_letters):
    """
    Filters a list of words from a CSV file, excluding words that contain
    specified letters.

    Args:
        filepath (str): The path to the CSV file containing the word list.
        absent_letters (str): A string containing the letters that should not
                              be present in the filtered words.

    Returns:
        list: A list of words that do not contain any of the specified absent
              letters. Returns an empty list if the file is not found or if
              there's an error.
    """
    try:
        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(filepath)

        # Check if the 'word' column exists
        if 'word' not in df.columns:
            print(f"Error: 'word' column not found in {filepath}")
            return []

        # Convert absent_letters to lowercase for case-insensitive comparison
        absent_letters = absent_letters.lower()

        # Filter words that do not contain any of the absent letters
        filtered_words = []
        for word in df['word']:
            word_lower = word.lower()
            valid = True
            for letter in absent_letters:
                if letter in word_lower:
                    valid = False
                    break
            if valid:
                filtered_words.append(word)

        return filtered_words

    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return []
    except pd.errors.EmptyDataError:
        print(f"Error: The file {filepath} is empty.")
        return []
    except pd.errors.ParserError:
        print(f"Error: Could not parse {filepath}. Please check the file format.")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []

def filter_words_by_present_letters_and_positions(word_list, present_letters, letter_positions='-----'):
    """
    Filters a list of words from a CSV file based on the presence of specific
    letters and their positions.

    Args:
        filepath (str): The path to the CSV file containing the word list.
        present_letters (str): A string containing the letters that MUST be
                               present in the filtered words.
        letter_positions (str): A string representing the required letter
                                positions. Use '_' to represent unknown
                                positions. For example, "__e__" means 'e'
                                must be in the third position. The length of
                                this string should match the word length.

    Returns:
        list: A list of words that meet the specified criteria. Returns an
              empty list if the file is not found or if there's an error.
    """
    try:
        # Convert inputs to lowercase for case-insensitive comparison
        present_letters = present_letters.lower()
        letter_positions = letter_positions.lower()

        word_length = len(letter_positions)

        # Validate the entries
        if len(present_letters) > word_length:
            print(f'Error: present_letters cannot be longer than {word_length} chars as defined by letter_positions. Returning word_list.')
            return word_list
        elif not all(c.isalpha() or c == '-' for c in letter_positions):
            print(f"Error: Invalid characters in letter_positions. Only letters and '_' are allowed. Returning word_list.")
            return word_list
        elif len(present_letters) == 0:
            print("No present letters specified. Returning word_list.")
            return word_list

        # Filter words
        filtered_words = []
        for word in word_list:
            word_lower = word.lower()

            # Check word length
            if len(word_lower) != word_length:
                continue

            # Check for present letters
            all_present = True

            for letter in present_letters:
                if letter not in word_lower:
                    all_present = False
                    break
            if not all_present:
                continue

            # Check letter positions
            position_match = True
            for i, char in enumerate(letter_positions):
                if char != '-':
                    if word_lower[i] != char:
                        position_match = False
                        break
            if not position_match:
                continue

            filtered_words.append(word)

        return filtered_words

    except Exception as e:
        print(f"An unexpected error occurred: {e}. Returning word_list.")
        return word_list



# Example usage:
filepath = './lexodle_list_5.csv'  # Replace with your file path
absent_letters = "caretbilswr"
present_letters = 'ong'
letter_positions = '-o-ng'
# positioning asst '12345'

filtered_words = filter_words_by_absent_letters(filepath, absent_letters)
filtered_words2 = filter_words_by_present_letters_and_positions(filtered_words, present_letters, letter_positions)


if filtered_words2:
    print(f"Words without the letters '{absent_letters}' & with the letters '{present_letters}'. Positions: '{letter_positions}':")
    print(f"Total words found: {len(filtered_words2)}")
    for word in filtered_words2:
        print(word)
else:
    print("No words found matching the criteria.")

Words without the letters 'caretbilswr' & with the letters 'ong'. Positions: '-o-ng':
Total words found: 1
young
